In [11]:
import brightway2 as bw
import pandas as pd
import openpyxl 
import os
import glob

In [12]:
bw.projects.set_current('Nitrous oxide production')

In [13]:
nitrousOxideDB = bw.Database('nitrous oxide')

In [14]:
GWPMethod = [method for method in bw.methods if 'IPCC 2013' in str(method) and  'climate change' 
                in str(method) and 'GWP 100a' in str(method) and not 'no LT' in str(method)]

In [15]:
def breakdown_calculations(db, activity):
    activitiesList = [{activity : 1}]
    for exchange in activity.technosphere():
        activitiesList.append({bw.Database(exchange.input.key[0]).get(exchange.input.key[1]) : exchange.amount})
    calculationSetup = {'inv' : activitiesList, 'ia' : GWPMethod}
    bw.calculation_setups['breakdown'] = calculationSetup
    myLCA = bw.MultiLCA('breakdown')
    results = pd.DataFrame(myLCA.results.transpose(), columns = [str(list(i.keys())[0]).split('\'')[1] for i in activitiesList], index = pd.MultiIndex.from_tuples(GWPMethod))
    results = results.sort_index().drop(index = [i for i in results.index if i[0] == 'rest'])
    directEmissions = pd.DataFrame([0 if abs(results.iloc[r,0] - results.iloc[r,1:].sum()) / abs(results.iloc[r,0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                      columns = ['direct emissions'],
                                      index = results.index)
    results = pd.concat([results, directEmissions], axis = 1)
    return results

In [16]:
resultsDF = pd.DataFrame()

In [17]:
nitrousOxideActivities = [activity for activity in nitrousOxideDB if 'nitrous oxide' == activity['reference product']]
nitrousOxideActivities

['nitrous oxide production; AON (100%); fossil ammonia' (kilogram, GLO, None),
 'nitrous oxide production; AON (100%); wind ammonia' (kilogram, GLO, None),
 'nitrous oxide production; BAU (no pollution); wind ammonia' (kilogram, GLO, None),
 'nitrous oxide production; BAU (pollution); wind ammonia' (kilogram, GLO, None),
 'nitrous oxide production; BAU (pollution); fossil ammonia' (kilogram, GLO, None),
 'nitrous oxide production; AON (86%); fossil ammonia' (kilogram, GLO, None),
 'nitrous oxide production; AON (86%); wind ammonia' (kilogram, GLO, None),
 'nitrous oxide production; BAU (no pollution); fossil ammonia' (kilogram, GLO, None)]

In [18]:
for nitrousOxideActivity in nitrousOxideActivities:
    resultsDF = pd.concat([resultsDF, breakdown_calculations(nitrousOxideDB, nitrousOxideActivity)], ignore_index = True)

In [19]:
resultsDF.fillna(0, inplace = True)
resultsDF

,nitrous oxide production; AON (100%); fossil ammonia,"ammonia production, steam reforming, liquid","market group for electricity, medium voltage",cooling water,direct emissions,nitrous oxide production; AON (100%); wind ammonia,ammonia production; hydrogen from PEM electrolysis; onshore wind electricity,nitrous oxide production; BAU (no pollution); wind ammonia,nitrous oxide production; BAU (pollution); wind ammonia,nitrous oxide production; BAU (pollution); fossil ammonia,nitrous oxide production; AON (86%); fossil ammonia,nitrous oxide production; AON (86%); wind ammonia,nitrous oxide production; BAU (no pollution); fossil ammonia
0,2.211188,1.971162,0.228191,0.011835,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.228191,0.011835,0.00000,0.914405,0.674379,0.00000,0.00000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.414999,0.006351,0.00000,0.000000,0.741929,1.16328,0.00000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.429625,0.006424,2.43616,0.000000,0.786761,0.00000,3.65897,0.000000,0.000000,0.000000,0.000000
4,0.000000,2.299647,0.429625,0.006424,2.43616,0.000000,0.000000,0.00000,0.00000,5.171856,0.000000,0.000000,0.000000
5,0.000000,2.292325,0.257442,0.013538,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,2.563306,0.000000,0.000000
6,0.000000,0.000000,0.257442,0.013538,0.00000,0.000000,0.784256,0.00000,0.00000,0.000000,0.000000,1.055237,0.000000
7,0.000000,2.168607,0.414999,0.006351,0.00000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000,2.589957


In [20]:
resultsFilePath = os.path.join('..', 'Results', 'GWP breakdown.xlsx')
with pd.ExcelWriter(resultsFilePath, engine='xlsxwriter') as writer:
    resultsDF.to_excel(writer, sheet_name = 'Overall')